In [0]:
import requests
import pandas as pd

url = "https://api-web.nhle.com/v1/club-schedule-season/MTL/20252026"
games = requests.get(url).json()['games']

df = pd.DataFrame([{
    'date': g['gameDate'],
    'game_type': g['gameType'],          # 2=regular, 3=playoff
    'home_team': g['homeTeam']['abbrev'],
    'away_team': g['awayTeam']['abbrev'],
    'is_home': g['homeTeam']['abbrev'] == 'MTL',
    'venue': g.get('venue', {}).get('default'),
    'start_utc': g['startTimeUTC'],
} for g in games])

df['date'] = pd.to_datetime(df['date'])
df['game_day'] = 1

# Playoffs only
playoffs = df[df['game_type'] == 3].copy()
print(playoffs[['date', 'home_team', 'away_team', 'is_home']])

In [0]:
import pyspark.sql.functions as F

T_trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_location = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")

In [0]:
revenue_df_1= (
    T_trans
    .filter(F.col("VisitDateId") > '20260101')
    .filter(F.col("VisitDateId") < '20260527')
    .join(T_location, T_trans.LocationId == T_location.LocationID, how="inner")
    .withColumn("day_of_week", F.dayofweek(F.to_date(F.col("VisitDateId"), "yyyyMMdd")))
    .withColumn("date", F.to_date(F.col("VisitDateId"), "yyyyMMdd"))
    .filter(F.col("Location_Name").like("%Royal%"))
    .groupBy("date")
    .agg(
        F.sum(F.col("Net_Revenue")).alias("revenue")
)   
)

revenue_df = revenue_df_1.toPandas()

In [0]:
revenue_df['date'] = pd.to_datetime(revenue_df['date'])
revenue_df['revenue'] = revenue_df['revenue'].astype(float)
revenue_df = revenue_df.sort_values('date').set_index('date')

# Check for missing dates in the range
full_range = pd.date_range(revenue_df.index.min(), revenue_df.index.max(), freq='D')
missing = full_range.difference(revenue_df.index)
print(f"Missing dates: {len(missing)}")
print(missing.tolist() if len(missing) < 20 else missing[:20].tolist())

# Reindex to fill any gaps (forward-fill or zero — decide based on what missing means)
revenue_df = revenue_df.reindex(full_range)
print(f"NaN revenue rows: {revenue_df['revenue'].isna().sum()}")

In [0]:
import pandas as pd
import numpy as np
from scipy import stats

# ---- Inputs assumed already in memory ----
# revenue_df: DatetimeIndex, column 'revenue'
# playoffs:   DataFrame with 'date' column (playoff game dates)
# df:         full Habs schedule (used to flag regular-season games too)

# 1. Build daily frame with game flags
daily = revenue_df.copy()
daily['dow'] = daily.index.dayofweek          # 0=Mon, 6=Sun
daily['month'] = daily.index.month

playoff_dates = set(pd.to_datetime(playoffs['date']).dt.normalize())
regseason_dates = set(pd.to_datetime(df.loc[df['game_type'] == 2, 'date']).dt.normalize())

daily['is_playoff_game'] = daily.index.normalize().isin(playoff_dates)
daily['is_regseason_game'] = daily.index.normalize().isin(regseason_dates)
daily['is_any_habs_game'] = daily['is_playoff_game'] | daily['is_regseason_game']

# 2. Build the control pool: non-Habs-game days only (clean baseline)
control_pool = daily[~daily['is_any_habs_game']].copy()

# 3. For each playoff game day, find matched non-game days
#    Match on: same day of week, within ±28 days


results = []
for game_date in sorted(playoff_dates):
    if game_date not in daily.index:
        continue
    target = daily.loc[game_date]
    target_dow = target['dow']
    
    window_start = game_date - pd.Timedelta(days=28)
    window_end   = game_date + pd.Timedelta(days=28)
    
    matches = control_pool[
        (control_pool['dow'] == target_dow) &
        (control_pool.index >= window_start) &
        (control_pool.index <= window_end)
    ]
    
    if len(matches) < 2:
        print(f"⚠ {game_date.date()}: only {len(matches)} matches — skipping")
        continue
    
    expected = float(matches['revenue'].mean())
    actual = float(target['revenue'])
    lift_pct = (actual / expected - 1) * 100
    
    # Also flag if it was a home game
    is_home = playoffs.loc[
        pd.to_datetime(playoffs['date']) == game_date, 'is_home'
    ].iloc[0]
    
results.append({
    'game_date': game_date.date(),
    'dow': game_date.strftime('%a'),
    'is_home': is_home,
    'n_matches': len(matches),

    'actual_revenue': actual,
    'expected_revenue': expected,
    'incremental_revenue': actual - expected,

    'lift_pct': lift_pct,
})
results_df = pd.DataFrame(results)

# 4. Aggregate stats
print("\n=== Per-game lift ===")
print(results_df.to_string(index=False))

print("\n=== Summary ===")
mean_lift = results_df['lift_pct'].mean()
median_lift = results_df['lift_pct'].median()

# One-sample t-test: is mean lift different from 0?
t_stat, p_value = stats.ttest_1samp(results_df['lift_pct'], 0)

# Bootstrap 95% CI on the mean
boot_means = [
    np.random.choice(results_df['lift_pct'], len(results_df), replace=True).mean()
    for _ in range(5000)
]
ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])

print("\n=== Per-game lift ===")
print(
    results_df[
        [
            'game_date',
            'dow',
            'is_home',
            'actual_revenue',
            'expected_revenue',
            'incremental_revenue',
            'lift_pct'
        ]
    ].to_string(index=False)
)

In [0]:
import pandas as pd
import numpy as np
from scipy import stats

# ---- Inputs assumed already in memory ----
# revenue_df: DatetimeIndex, column 'revenue'
# playoffs:   DataFrame with 'date' column (playoff game dates)
# df:         full Habs schedule (used to flag regular-season games too)

# 1. Build daily frame with game flags
daily = revenue_df[['revenue']].copy()

daily['dow'] = daily.index.dayofweek          # 0=Mon, 6=Sun
daily['month'] = daily.index.month

playoff_dates = set(pd.to_datetime(playoffs['date']).dt.normalize())
regseason_dates = set(
    pd.to_datetime(
        df.loc[df['game_type'] == 2, 'date']
    ).dt.normalize()
)

daily['is_playoff_game'] = daily.index.normalize().isin(playoff_dates)
daily['is_regseason_game'] = daily.index.normalize().isin(regseason_dates)
daily['is_any_habs_game'] = (
    daily['is_playoff_game'] |
    daily['is_regseason_game']
)

# 2. Build the control pool: non-Habs-game days only
control_pool = daily[~daily['is_any_habs_game']].copy()

# 3. Match playoff days to comparable non-game days
#    Match on:
#    - same day of week
#    - within ±28 days

results = []

for game_date in sorted(playoff_dates):

    if game_date not in daily.index:
        continue

    target = daily.loc[game_date]

    target_dow = target['dow']

    window_start = game_date - pd.Timedelta(days=28)
    window_end   = game_date + pd.Timedelta(days=28)

    matches = control_pool[
        (control_pool['dow'] == target_dow) &
        (control_pool.index >= window_start) &
        (control_pool.index <= window_end)
    ]

    if len(matches) < 2:
        print(f"⚠ {game_date.date()}: only {len(matches)} matches — skipping")
        continue

    expected = float(matches['revenue'].mean())
    actual = float(target['revenue'])

    incremental = actual - expected

    lift_pct = (actual / expected - 1) * 100

    # Home / away flag
    is_home = playoffs.loc[
        pd.to_datetime(playoffs['date']).dt.normalize() == game_date,
        'is_home'
    ].iloc[0]

    results.append({
        'game_date': game_date.date(),
        'dow': game_date.strftime('%a'),
        'is_home': is_home,
        'n_matches': len(matches),

        'actual_revenue': round(actual, 2),
        'expected_revenue': round(expected, 2),
        'incremental_revenue': round(incremental, 2),

        'lift_pct': round(lift_pct, 2),
    })

# 4. Results dataframe
results_df = pd.DataFrame(results)

# 5. Per-game output
print("\n=== Per-game lift ===")

if len(results_df) > 0:

    print(
        results_df[
            [
                'game_date',
                'dow',
                'is_home',
                'n_matches',
                'actual_revenue',
                'expected_revenue',
                'incremental_revenue',
                'lift_pct'
            ]
        ].to_string(index=False)
    )

    # 6. Summary stats
    print("\n=== Summary ===")

    mean_lift = results_df['lift_pct'].mean()
    median_lift = results_df['lift_pct'].median()

    total_incremental = results_df['incremental_revenue'].sum()

    print(f"Mean lift:                {mean_lift:+.2f}%")
    print(f"Median lift:              {median_lift:+.2f}%")
    print(f"Total incremental rev:    ${total_incremental:,.2f}")

    # Statistical tests require >= 2 observations
    if len(results_df) >= 2:

        t_stat, p_value = stats.ttest_1samp(
            results_df['lift_pct'],
            0
        )

        boot_means = [
            np.random.choice(
                results_df['lift_pct'],
                len(results_df),
                replace=True
            ).mean()
            for _ in range(5000)
        ]

        ci_low, ci_high = np.percentile(
            boot_means,
            [2.5, 97.5]
        )

        print(f"95% CI:                  [{ci_low:+.2f}%, {ci_high:+.2f}%]")
        print(f"p-value:                 {p_value:.4f}")

    else:
        print("Not enough observations for t-test / confidence interval")

    # 7. Home vs Away
    print("\n=== Home vs Away ===")

    print(
        results_df.groupby('is_home')['lift_pct']
        .agg(['mean', 'median', 'count'])
    )

else:
    print("No valid playoff games available after matching.")

In [0]:
results_df.display()

In [0]:
std_dev_lift = results_df['lift_pct'].std()

std_dev_incremental = results_df['incremental_revenue'].std()

print(f"Lift Std Dev:             {std_dev_lift:.2f}%")
print(f"Incremental Rev Std Dev:  ${std_dev_incremental:,.2f}")

In [0]:
boot_incrementals = [
    np.random.choice(
        results_df['incremental_revenue'],
        len(results_df),
        replace=True
    ).mean()
    for _ in range(5000)
]

inc_ci_low, inc_ci_high = np.percentile(
    boot_incrementals,
    [2.5, 97.5]
)

print(
    f"Incremental Revenue 95% CI: "
    f"[${inc_ci_low:,.0f}, ${inc_ci_high:,.0f}]"
)

In [0]:
avg_incremental_per_game = results_df['incremental_revenue'].mean()

print(
    f"Average Incremental Revenue Per Game: "
    f"${avg_incremental_per_game:,.0f}"
)

In [0]:
# 1. Check if the away-vs-home split is statistically meaningful
from scipy import stats
home = results_df.loc[results_df['is_home'], 'lift_pct']
away = results_df.loc[~results_df['is_home'], 'lift_pct']
t, p = stats.ttest_ind(away, home, equal_var=False)
print(f"Away vs Home: away mean {away.mean():+.1f}% vs home {home.mean():+.1f}%, p={p:.3f}")

# 2. Robust summary (less outlier-sensitive)
print(f"Trimmed mean (drop top/bottom): {stats.trim_mean(results_df['lift_pct'], 0.1):+.1f}%")

# 3. Wilcoxon signed-rank (non-parametric, doesn't assume normality)
w, p = stats.wilcoxon(results_df['lift_pct'])
print(f"Wilcoxon p-value: {p:.4f}")

In [0]:
import matplotlib.pyplot as plt
import numpy as np

AWAY_COLOR = '#1D9E75'
HOME_COLOR = '#D85A30'

# results_df should have: game_date, dow, is_home, lift_pct
plot_df = results_df.copy()
plot_df['label'] = pd.to_datetime(plot_df['game_date']).dt.strftime('%b %d') + '\n' + plot_df['dow']
plot_df['color'] = np.where(plot_df['is_home'], HOME_COLOR, AWAY_COLOR)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [3, 1]})

# --- Left: per-game lift bars ---
ax1 = axes[0]
ax1.bar(plot_df['label'], plot_df['lift_pct'], color=plot_df['color'], edgecolor='none')
ax1.axhline(0, color='#444', linewidth=0.8)
ax1.set_ylabel('Lift vs matched non-game days (%)', fontsize=11)
ax1.set_title('Per-game revenue lift — Rec Room Royalmount', fontsize=12, loc='left', pad=12)
ax1.tick_params(axis='x', labelsize=9)
ax1.grid(axis='y', alpha=0.2)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Value labels on bars
for i, v in enumerate(plot_df['lift_pct']):
    offset = 4 if v >= 0 else -4
    va = 'bottom' if v >= 0 else 'top'
    ax1.text(i, v + offset, f'{v:+.0f}%', ha='center', va=va, fontsize=8.5)

# Legend
from matplotlib.patches import Patch
ax1.legend(handles=[
    Patch(color=AWAY_COLOR, label='Away game'),
    Patch(color=HOME_COLOR, label='Home game'),
], loc='upper left', frameon=False, fontsize=10)

# --- Right: home vs away summary ---
# --- Right: home vs away summary ---
ax2 = axes[1]
summary = plot_df.groupby('is_home')['lift_pct'].agg(['mean', 'count']).reset_index()
summary['venue'] = np.where(summary['is_home'], 'Home', 'Away')
summary['color'] = np.where(summary['is_home'], HOME_COLOR, AWAY_COLOR)
summary = summary.sort_values('is_home')  # Away first, Home second

bars = ax2.bar(summary['venue'], summary['mean'], color=summary['color'],
               edgecolor='none', width=0.45)
ax2.axhline(0, color='#444', linewidth=0.8)
ax2.set_ylabel('Mean lift (%)', fontsize=11)
ax2.set_title('Away vs home', fontsize=12, loc='left', pad=12)
ax2.grid(axis='y', alpha=0.2)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Add headroom so labels don't collide with title or axis
y_max = max(summary['mean'].max(), 0)
y_min = min(summary['mean'].min(), 0)
pad = (y_max - y_min) * 0.25
ax2.set_ylim(y_min - pad, y_max + pad)

for bar, row in zip(bars, summary.itertuples()):
    h = bar.get_height()
    if h >= 0:
        ax2.text(bar.get_x() + bar.get_width()/2, h + pad*0.15,
                 f'{h:+.0f}%', ha='center', va='bottom',
                 fontsize=11, fontweight='bold')
        ax2.text(bar.get_x() + bar.get_width()/2, h + pad*0.55,
                 f'n={row.count}', ha='center', va='bottom',
                 fontsize=9, color='#666')
    else:
        ax2.text(bar.get_x() + bar.get_width()/2, h - pad*0.15,
                 f'{h:+.0f}%', ha='center', va='top',
                 fontsize=11, fontweight='bold')
        ax2.text(bar.get_x() + bar.get_width()/2, h - pad*0.55,
                 f'n={row.count}', ha='center', va='top',
                 fontsize=9, color='#666')

plt.suptitle('Habs playoffs revenue lift through 11 games  ·  Median +12%  ·  Mean +30%  ·  p=0.17',
             fontsize=11, y=1.02, x=0.02, ha='left', color='#555')
plt.tight_layout()
plt.show()